<a href="https://colab.research.google.com/github/Steph-business/Tech_Talent_Accelerator/blob/main/Week6_Exercises_XP_Day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercices XP: Jour 3 - BERT en Pratique


## Exercice 1 - Tokenisation avec BERT
Objectif: Explorer comment le tokenizer `bert-base-uncased` prépare le texte pour l'entrée du modèle.

Instructions:
1. (Optionnel) Installez les bibliothèques requises.
2. Chargez le tokenizer, créez une phrase d'exemple et encodez-la avec du padding et de la troncation.
3. Affichez les jetons à côté de leurs IDs entiers et signalez les jetons spéciaux.
4. Inspectez le masque d'attention pour voir comment le padding est masqué du modèle.

Livrables:
- TODO: Fournissez la liste imprimée des jetons et des IDs avec [CLS]/[SEP]/[PAD] mis en évidence.
- TODO: Documentez le choix de padding que vous avez fait et pourquoi il correspond à la longueur de la phrase.

In [ ]:
# Configuration optionnelle: installez les dépendances si elles sont manquantes dans votre environnement.
# %pip install -q transformers torch

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "L'apprentissage automatique avec BERT est absolument passionnant !"
print(f"Phrase choisie : {sample_sentence}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Phrase choisie : L'apprentissage automatique avec BERT est absolument passionnant !


In [ ]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: ajustez si votre phrase nécessite plus d'espace
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | jeton        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nMasque d'attention:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Jetons spéciaux (index, jeton):", special_positions)

### Réflexion sur l'Exercice 1
- **[CLS] et [SEP]** : Le jeton `[CLS]` (Classification) est inséré au début de la séquence et sert de représentation globale pour les tâches de classification. Le jeton `[SEP]` (Separator) sert à marquer la fin d'une phrase ou la séparation entre deux segments de texte.
- **Masque d'attention** : Il contient des 1 pour les jetons réels et des 0 pour les jetons de remplissage (`[PAD]`). Cela indique au mécanisme d'attention de ne pas tenir compte des espaces vides lors du calcul des relations contextuelles.

## Exercice 2 - Pipeline d'analyse de sentiment
Objectif: Utiliser un pipeline de sentiment DistilBERT pré-entraîné pour classifier une phrase.

Instructions:
1. Importez l'aide `pipeline` de `transformers`.
2. Construisez un pipeline qui charge `distilbert-base-uncased-finetuned-sst-2-english`.
3. Passez une phrase et examinez l'étiquette et le score prédits.

Livrables:
- TODO: Enregistrez la phrase que vous avez testée.
- TODO: Capturez l'étiquette et le score de confiance et interprétez le résultat.

In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "This tutorial is extremely helpful and easy to follow!"
prediction = sentiment_pipeline(sentence)
print(f"Phrase : {sentence}")
print(f"Résultat : {prediction}")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Phrase : This tutorial is extremely helpful and easy to follow!
Résultat : [{'label': 'POSITIVE', 'score': 0.9995120763778687}]


### Réflexion sur l'Exercice 2
- **Attente** : Le résultat 'POSITIVE' est tout à fait cohérent puisque la phrase contient des termes mélioratifs comme 'helpful' (utile) et 'easy' (facile).
- **Confiance** : Un score de confiance élevé (souvent > 0.99) démontre que le modèle DistilBERT, affiné sur le jeu de données SST-2, identifie très clairement la polarité positive de ce texte.

## Exercice 3 - Classe d'analyse de sentiment personnalisée
Objectif: Reconstruire le pipeline manuellement afin de contrôler la tokenisation, les tenseurs et le scoring.

Instructions:
1. Importez `AutoTokenizer` et `AutoModelForSequenceClassification`.
2. Implémentez `BERTSentimentAnalyzer` avec des méthodes pour l'initialisation, le prétraitement et la prédiction.
3. Testez la classe avec plusieurs phrases.

Conseils:
- Gardez un attribut `max_length` afin de pouvoir le réutiliser lors de la tokenisation.
- Appliquez `torch.softmax` pour transformer les logits en probabilités.
- Retournez à la fois l'étiquette et la probabilité pour plus de clarté.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.max_length = max_length

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        return self.tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length
        ).to(self.device)

    def predict(self, text: str) -> Dict[str, any]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = F.softmax(outputs.logits, dim=-1)
            confidence, class_idx = torch.max(probs, dim=-1)

        label = self.model.config.id2label[class_idx.item()]
        return {"label": label, "probability": confidence.item()}

In [ ]:
analyzer = BERTSentimentAnalyzer()
samples = [
    "I love how BERT handles context so well!",
    "This implementation is quite difficult and confusing."
]
for text in samples:
    res = analyzer.predict(text)
    print(f"Texte: {text}\nRésultat: {res}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Texte: I love how BERT handles context so well!
Résultat: {'label': 'POSITIVE', 'probability': 0.9998438358306885}

Texte: This implementation is quite difficult and confusing.
Résultat: {'label': 'NEGATIVE', 'probability': 0.9993501305580139}



## Exercice 4 - BERT pour la reconnaissance d'entités nommées
Objectif: Construire une classe légère qui exécute un modèle de classification de jetons et mappe les jetons à des étiquettes d'entités.

Instructions:
1. Importez `AutoTokenizer` et `AutoModelForTokenClassification`.
2. Implémentez `BERTNamedEntityRecognizer` avec l'initialisation et une méthode `recognize`.
3. Tokenisez un exemple de texte, exécutez le modèle, convertissez les prédictions en étendues d'entités et testez avec un court paragraphe.

Livrables:
- TODO: Retournez une liste de dictionnaires comme `{text, entity, start, end}` pour chaque entité détectée.
- TODO: Expliquez comment vous avez géré les sous-mots qui commencent par `##`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name).to(self.device)

    def recognize(self, text: str):
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.model(**inputs)

        predictions = torch.argmax(outputs.logits, dim=-1)
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

        entities = []
        for token, prediction in zip(tokens, predictions[0].tolist()):
            label = self.model.config.id2label[prediction]
            if label != "O":
                entities.append({"token": token, "label": label})
        return entities

In [ ]:
# Assurez-vous d'avoir exécuté la cellule 6aa24c9a avant celle-ci
ner = BERTNamedEntityRecognizer()
sample_text = "Elon Musk is the CEO of SpaceX, which aims to send humans to Mars."
entities = ner.recognize(sample_text)

print(f"Texte analysé : {sample_text}\n")
print("Entités détectées :")
for ent in entities:
    print(f"- {ent['token']:<10} : {ent['label']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Texte analysé : Elon Musk is the CEO of SpaceX, which aims to send humans to Mars.

Entités détectées :
- El         : B-PER
- ##on       : B-PER
- Mu         : I-PER
- ##sk       : I-PER
- Space      : B-ORG
- ##X        : I-ORG
- Mars       : B-LOC


### Analyse des résultats du NER

Le modèle BERT-NER a produit les résultats suivants :
1. **Segmentation en sous-mots** : Vous remarquerez des jetons comme `##on`, `Mu` et `##sk`. BERT utilise un tokenizer **WordPiece** qui découpe les mots inconnus ou complexes en morceaux plus petits pour mieux les traiter.
2. **Étiquettes d'entités** :
   - `B-PER` / `I-PER` : Début et intérieur d'un nom de **Personne** (Elon Musk).
   - `B-ORG` / `I-ORG` : Début et intérieur d'une **Organisation** (SpaceX).
   - `B-LOC` : Un **Lieu** (Mars).

Le modèle a réussi à capturer le contexte sémantique de la phrase pour classifier ces entités avec précision.

## Exercice 5 - Comparaison de BERT et GPT

| Catégorie | BERT | GPT |
|----------|------|-----|
| Architecture | Encodeur Transformer (Bidirectionnel) | Décodeur Transformer (Unidirectionnel / Causal) |
| Objectif principal | Compréhension du langage (NLU) | Génération de texte (NLG) |
| Cas d'utilisation | Classification, NER, Recherche sémantique | Chatbots, Rédaction, Complétion de code |
| Points forts | Compréhension profonde du contexte global | Fluidité narrative et capacités génératives |
| Points faibles | Moins efficace pour la génération de texte long | Ne peut pas 'voir' le contexte futur (unidirectionnel) |

## Exercice 6 - BERT au sein du RAG (Génération Augmentée par Récupération)

1. **Encodage** : BERT transforme les questions des utilisateurs et les documents sources en vecteurs denses (embeddings). Contrairement à une recherche par mots-clés, cela capture le sens profond de la requête.
2. **Recherche vectorielle** : Ces vecteurs sont stockés dans une base de données vectorielle (comme FAISS ou Pinecone). On effectue une recherche de similarité cosinus pour trouver les documents les plus proches de la question.
3. **Passage au modèle génératif** : Les passages les plus pertinents récupérés grâce à BERT sont fournis en contexte à un modèle de type GPT, qui s'en sert pour formuler une réponse précise et factuelle.
4. **Exemple concret** : Dans un centre d'aide technique, BERT peut retrouver la section exacte d'un manuel complexe pour qu'un agent conversationnel explique à l'utilisateur comment réparer sa machine.